In [7]:
import pandas as pd
from pathlib import Path
import re
import numpy as np

# Use era5_uk_local.py and era5_usa_utc.py to get the correct timezoning for era5. 
era_path = Path("/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/")


## Getting rid of electricity outliers!
Helper function

In [5]:
def qc_electricity(df_long, has_sectors=True):
    df = df_long.copy().sort_values(['Sector', 'Date'] if has_sectors else ['Date'])
    #First remove any zeros
    df.loc[df['Demand'] == 0, 'Demand'] = np.nan

    # get the 1st and 3rd quantiles, flag anything higher than q3 + 3*iqr and anything lower than q1 - 3*iqr
    def flag_level_outliers(series):
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 3 * iqr
        upper = q3 + 3 * iqr
        return (series < lower) | (series > upper)

    #Flag if usage jumps by over 50%
    def flag_jump_outliers(series):
        pct = series.pct_change(fill_method=None).abs()
        return (pct > 0.50) | (series == 0)

    if has_sectors:
        level_flags = df.groupby('Sector')['Demand'].transform(flag_level_outliers)
        jump_flags  = df.groupby('Sector')['Demand'].transform(flag_jump_outliers)
    else:
        level_flags = flag_level_outliers(df['Demand'])
        jump_flags  = flag_jump_outliers(df['Demand'])

    flagged = level_flags | jump_flags
    n_flagged = flagged.sum()
    print(f"Flagged {n_flagged} rows ({100 * n_flagged / len(df):.2f}%) as anomalous — setting to NaN")

    # ---- show examples ----
    flagged_rows = df[flagged].copy()
    flagged_rows['flag_type'] = np.where(
        level_flags[flagged], 
        np.where(jump_flags[flagged], 'both', 'level'), 
        'jump'
    )
    print("\nSample flagged values (up to 20):")
    if has_sectors: print(flagged_rows[['Sector', 'Date', 'Demand', 'flag_type']].head(20).to_string(index=False))
    else: print(flagged_rows[['Date', 'Demand', 'flag_type']].head(20).to_string(index=False))

    if has_sectors:
        print("\nFlagged counts by sector:")
        # use df index to avoid the reindex warning
        by_sector = flagged_rows.groupby('Sector').size()
        if not by_sector.empty:
            print(by_sector.to_string())

    df.loc[flagged, 'Demand'] = np.nan
    return df

# 1. USA

In [52]:
data_path = Path("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/USA_raw/")
dfs = []
for file in sorted(data_path.glob("eia_hourly_*.csv")):
    df = pd.read_csv(file)
    # Parse datetime
    df["Date"] = pd.to_datetime(df["period"])
    # Aggregate sub-sectors → parent totals per hour
    df_parent = (df.groupby(["parent", "Date"], as_index=False)["value"].sum())
    # Rename columns to final schema
    df_parent = df_parent.rename(columns={"parent": "Sector", "value": "Demand"})
    dfs.append(df_parent)
# Combine all years
usa_all = pd.concat(dfs, ignore_index=True)
usa_all = usa_all.sort_values(["Sector", "Date"])

#QC!
usa_all = qc_electricity(usa_all, has_sectors=True)

# Save
usa_all.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/usa_hourly.csv",index=False)
print('done!')

Flagged 333 rows (0.08%) as anomalous — setting to NaN

Sample flagged values (up to 20):
Sector                Date  Demand flag_type
  CISO 2019-08-15 01:00:00 43597.0     level
  CISO 2019-08-15 02:00:00 43286.0     level
  CISO 2019-08-16 00:00:00 43185.0     level
  CISO 2019-08-16 01:00:00 43849.0     level
  CISO 2019-08-16 02:00:00 43281.0     level
  CISO 2019-08-27 01:00:00 43279.0     level
  CISO 2019-09-04 00:00:00 43069.0     level
  CISO 2019-09-04 01:00:00 43731.0     level
  CISO 2019-09-04 02:00:00 43160.0     level
  CISO 2019-09-04 23:00:00 43181.0     level
  CISO 2019-09-05 00:00:00 43074.0     level
  CISO 2020-08-14 22:00:00 43203.0     level
  CISO 2020-08-14 23:00:00 44942.0     level
  CISO 2020-08-15 00:00:00 45909.0     level
  CISO 2020-08-15 01:00:00 46024.0     level
  CISO 2020-08-15 02:00:00 45345.0     level
  CISO 2020-08-15 03:00:00 42990.0     level
  CISO 2020-08-15 23:00:00 43048.0     level
  CISO 2020-08-16 00:00:00 43793.0     level
  CISO 202

In [29]:
# Now let's link it to the climate data! 
electric = pd.read_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/USA_hourly.csv")
dfs = []

for file in sorted(era_path.glob("*hourly*USA*utc*")):
    print(file)
    df = pd.read_csv(file, encoding="latin1")
    dfs.append(df)
era = pd.concat(dfs, ignore_index=True)
era.columns = era.columns.str.strip()
electric.columns = electric.columns.str.strip()
# Ensure datetime alignment
era['Date'] = pd.to_datetime(era['time_utc'])
electric['Date'] = pd.to_datetime(electric['Date'])
df_merged = pd.merge(electric, era.rename(columns={'EIAcode': 'Sector'}), on=['Date', 'Sector'], how='inner')
df_merged = df_merged[['Date', 'Sector', 'Demand', 'T', 'Td', 'RH', 'HI', 'Tw', 'HU', 'Q']]
# Save result
df_merged.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/USA_data.csv", index=False)
print('done!')

/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/._era5_hourly_USA_2022_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_hourly_USA_2019_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_hourly_USA_2020_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_hourly_USA_2021_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_hourly_USA_2022_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_hourly_USA_2023_utc.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly

In [5]:
# Now aggregate to daily + link to daily era5!

electric = pd.read_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/USA_hourly.csv")
electric['Date'] = pd.to_datetime(electric['Date'])
electric['Date'] = electric['Date'].dt.floor('D')
# Sum hourly demand → daily demand per sector
electric['Date_day'] = pd.to_datetime(electric['Date']).dt.floor('D')

# filter so we only include days with all hours present!
electric_daily = (
    electric.groupby(['Date_day', 'Sector'])
    .agg(
        Demand=('Demand', 'sum'),
        hours_present=('Demand', 'count'),   # counts non-NaN only
        hours_null=('Demand', lambda x: x.isna().sum())
    )
    .reset_index()
    .rename(columns={'Date_day': 'Date'})
)

electric_daily = electric_daily[electric_daily['hours_present'] == 24].drop(columns=['hours_present', 'hours_null'])

print('got electric')
#NOW: link to era5! 

dfs = []
for file in sorted(era_path.glob("*daily_USA*utc2*")):
    print(file)
    df = pd.read_csv(file, encoding="latin1")
    dfs.append(df)

print('read files')

era = pd.concat(dfs, ignore_index=True)
era.columns = era.columns.str.strip()

# Ensure datetime
era['Date'] = pd.to_datetime(era['date_utc']).dt.floor('D')
#era = era.rename(columns={'EIAcode': 'Sector'})

era = era.rename(columns={
    'T_mean': 'T',
    'Td_mean': 'Td',
    'HI_mean': 'HI',
    'HU_mean': 'HU',
    'RH_mean': 'RH',
    'Tw_mean': 'Tw',
    'Q_mean': 'Q',
    'cdd': 'CDD',
    'cdd_HI': 'CDDhi',
    'cdd_HU': 'CDDhu',
    'cdd_Tw': 'CDDtw',
    'cdd_Q': 'CDDq'  
})

df_merged = pd.merge(
    electric_daily,
    era,
    on=['Date', 'Sector'],
    how='inner'
)

df_merged.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/USA_daily.csv", index=False)
print('Done!')

got electric
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2019_utc2.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2020_utc2.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2021_utc2.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2022_utc2.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2023_utc2.csv
/Users/charlottecockburn/Documents/Columbia/AC_project/remote_deela/Humidity_Paper/timezone_conversions/hourly_data_tz/era5_daily_USA_2024_utc2.csv
read files
Done!


## 2. United Kingdom

In [2]:
import pandas as pd

# Per-year resource IDs (the resource ID in the original URL is fixed to the
# *current* year's file regardless of the year in the filename, so each year
# needs its own resource ID)
UK_RESOURCE_IDS = {
    2019: "dd9de980-d724-415a-b344-d8ae11321432",
    2020: "33ba6857-2a55-479f-9308-e5c4c53d4381",
    2021: "18c69c42-f20d-46f0-84e9-e279045befc6",
    2022: "bb44a1b5-75b1-4db2-8491-257f23385006",
    2023: "bf5ab335-9b40-4ea4-b93a-ab4af7bce003",
    2024: "f6d02c0f-957b-48cb-82ee-09003f2ba759",
}

dfs = []
for year, resource_id in UK_RESOURCE_IDS.items():
    url = f"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/{resource_id}/download/demanddata_{year}.csv"
    try:
        df = pd.read_csv(url)
        dfs.append(df)
        print(f"Downloaded {year}")
    except Exception as e:
        print(f"Failed {year}: {e}")

uk_demand = pd.concat(dfs, ignore_index=True)
print(uk_demand.columns.tolist())  # check column names


Downloaded 2019
Downloaded 2020
Downloaded 2021
Downloaded 2022
Downloaded 2023
Downloaded 2024
['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND', 'TSD', 'ENGLAND_WALES_DEMAND', 'EMBEDDED_WIND_GENERATION', 'EMBEDDED_WIND_CAPACITY', 'EMBEDDED_SOLAR_GENERATION', 'EMBEDDED_SOLAR_CAPACITY', 'NON_BM_STOR', 'PUMP_STORAGE_PUMPING', 'IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW', 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW', 'NSL_FLOW', 'ELECLINK_FLOW', 'VIKING_FLOW', 'GREENLINK_FLOW', 'SCOTTISH_TRANSFER']


In [8]:
# National Demand (ND, in MW) is a half-hourly average; SETTLEMENT_DATE/PERIOD are
# Europe/London local clock time (46 periods on spring-forward days, 50 on fall-back)
df = uk_demand[["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "ND"]].copy()
df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"], format="mixed", dayfirst=True)

# Local midnight is never ambiguous in the UK, so localize the date then add the
# settlement-period offset as real elapsed time to get the correct UTC instant
local_midnight_utc = df["SETTLEMENT_DATE"].dt.tz_localize("Europe/London").dt.tz_convert("UTC")
utc_time = local_midnight_utc + pd.to_timedelta((df["SETTLEMENT_PERIOD"] - 1) * 30, unit="min")
df["Date"] = utc_time.dt.tz_convert("Europe/London").dt.tz_localize(None)

# Average the two half-hourly MW readings into hourly average MW (the two repeated
# half-hours on the fall-back day fold into a single hourly value)
df["Date"] = df["Date"].dt.floor("h")
uk_hourly = df.groupby("Date", as_index=False)["ND"].mean().rename(columns={"ND": "Demand"})

# Keep 2019-2024 for comparability with the USA series
uk_hourly = uk_hourly[(uk_hourly["Date"].dt.year >= 2019) & (uk_hourly["Date"].dt.year <= 2024)]

uk_hourly = qc_electricity(uk_hourly, has_sectors=False)

uk_hourly.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/uk_hourly.csv", index=False)
print('Done!')


Flagged 0 rows (0.00%) as anomalous — setting to NaN

Sample flagged values (up to 20):
Empty DataFrame
Columns: [Date, Demand, flag_type]
Index: []
Done!


In [10]:
# Now let's link it to the climate data!
electric = pd.read_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/uk_hourly.csv")
dfs = []
for file in sorted(era_path.glob("*hourly_United_Kingdom*")):
    df = pd.read_csv(file, encoding="latin1")
    dfs.append(df)
era = pd.concat(dfs, ignore_index=True)
# Clean column names (VERY important after CSVs)
era.columns = era.columns.str.strip()
electric.columns = electric.columns.str.strip()
# Ensure datetime alignment
era['Date'] = pd.to_datetime(era['time_local'])
electric['Date'] = pd.to_datetime(electric['Date'])
df_merged = pd.merge(electric, era, on=['Date'], how='inner')
df_merged = df_merged[['Date', 'Demand', 'T', 'Td', 'RH', 'HI', 'Tw', 'HU', 'Q']]
# Save result
df_merged.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/UK_data.csv", index=False)
print('done!')


done!


In [11]:
# Now daily!!!

electric = pd.read_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/Subnational_Electricity/uk_hourly.csv")
electric['Date'] = pd.to_datetime(electric['Date'])
electric['Date'] = electric['Date'].dt.floor('D')

electric['Date_day'] = pd.to_datetime(electric['Date']).dt.floor('D')
has_sectors = 'Sector' in electric.columns
group_cols = ['Date_day', 'Sector'] if has_sectors else ['Date_day']
electric_daily = (
    electric.groupby(group_cols)
    .agg(
        Demand=('Demand', 'sum'),
        hours_present=('Demand', 'count'),
        hours_null=('Demand', lambda x: x.isna().sum())
    )
    .reset_index()
    .rename(columns={'Date_day': 'Date'})
)
electric_daily = electric_daily[electric_daily['hours_present'] == 24].drop(columns=['hours_present', 'hours_null'])

dfs = []
for file in sorted(era_path.glob("era5_daily_United_Kingdom*")):
    df = pd.read_csv(file, encoding="latin1")
    dfs.append(df)

era = pd.concat(dfs, ignore_index=True)
era.columns = era.columns.str.strip()

# Ensure datetime
era['Date'] = pd.to_datetime(era['date_local']).dt.floor('D')

df_merged = pd.merge(
    electric_daily,
    era,
    on='Date',
    how='inner'
)

df_merged.to_csv("/Users/charlottecockburn/Documents/Columbia/AC_project/Data/UK_data_daily.csv", index=False)
print('Done!')


Done!
